# Building a 1-D CNN (Temporal Convolution Network) Classifier

> Imports


In [1]:
%load_ext autoreload
%autoreload 2

import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from helpers import plotLightCurveFromDF, sampleRandomKIC, saveLightCurveFromDF
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset

%matplotlib inline


/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


> Set model training device.
> For Apple Silicon, the device is mps.

In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)


mps


---

In [ ]:
df = pd.read_parquet("./local-assets/MERGED_LCS.parquet")
df


In [ ]:
df = df.reset_index()


---

> Rename features


In [ ]:
renameMap = {
    'Class': 'class',
}

df = df.rename(columns=renameMap)
df


---

> Remove duplicate KIC entries

In [ ]:
def chooseLabel(labels):
    labels = list(labels)
    nonFp = [lab for lab in labels if lab != "FALSE POSITIVE"]

    if len(nonFp) > 0:
        # If there are multiple non-FP labels, take the most common one
        return pd.Series(nonFp).mode().iloc[0]

    # If all labels are FALSE POSITIVE, keep one of them
    return pd.Series(labels).mode().iloc[0]


In [ ]:
df = (
    df.groupby("KIC", as_index=False)
        .agg({
            "time": "first",
            "flux": "first",
            "class": chooseLabel
        })
)


In [ ]:
df['class'].value_counts()


In [ ]:
df.duplicated(subset="KIC").sum()


---

> Remove 'CANDIDATE' class

In [ ]:
df = df[df['class'] != 'CANDIDATE'].copy()


In [ ]:
df


---

> Encode 'Class' labels

In [ ]:
def encodeLabels(df):
    """
    Function to encode the labels in the 'Class' feature of a given dataframe df.
    """
    le = LabelEncoder()
    y = le.fit_transform(df["class"].values)
    df = df.copy()
    df["label"] = y

    return df, le


In [ ]:
df, le = encodeLabels(df)
df


---

> Light curve raw data preprocessing


In [ ]:
def cleanTimeFlux(time, flux):
    """
    Convert inputs to arrays, flatten them, and keep only paired finite values.
    """
    time = np.asarray(time, dtype=np.float32).ravel()
    flux = np.asarray(flux, dtype=np.float32).ravel()

    if len(time) != len(flux):
        raise ValueError(f"time and flux have different lengths: {len(time)} vs {len(flux)}")

    mask = np.isfinite(time) & np.isfinite(flux)
    time = time[mask]
    flux = flux[mask]

    if len(time) == 0:
        raise ValueError("No valid time-flux pairs after removing NaN/inf values.")

    return time, flux


In [ ]:
def normalizeFlux(flux):
    """
    Normalize flux after time/flux cleaning.
    """
    flux = np.asarray(flux, dtype=np.float32).ravel()

    if len(flux) == 0:
        raise ValueError("Empty flux array after cleaning.")

    med = np.median(flux)
    if not np.isfinite(med) or med == 0:
        med = 1.0

    flux = flux / med - 1.0
    return flux


In [ ]:
def resampleToFixedLength(time, flux, seq_len=2000):
    """
    Resample cleaned, aligned time-flux pairs to a fixed sequence length.
    """
    time = np.asarray(time, dtype=np.float32).ravel()
    flux = np.asarray(flux, dtype=np.float32).ravel()

    if len(time) != len(flux):
        raise ValueError(f"time and flux must match in length: {len(time)} vs {len(flux)}")

    if len(time) < 2:
        raise ValueError("Not enough valid points to resample.")

    order = np.argsort(time)
    time = time[order]
    flux = flux[order]

    # Remove duplicate timestamps
    uniq_time, uniq_idx = np.unique(time, return_index=True)
    time = uniq_time
    flux = flux[uniq_idx]

    if len(time) < 2:
        raise ValueError("Not enough unique time points.")

    t_new = np.linspace(time.min(), time.max(), seq_len)
    y_new = np.interp(t_new, time, flux).astype(np.float32)

    return y_new


In [ ]:
def preprocessLightCurve(time, flux, seq_len=2000, clip_sigma=5.0):
    """
    Full preprocessing pipeline:
    1) clean paired time/flux values
    2) normalize flux
    3) clip outliers
    4) resample to fixed length
    5) standardize
    """
    time, flux = cleanTimeFlux(time, flux)
    flux = normalizeFlux(flux)

    mu = np.mean(flux)
    sd = np.std(flux)
    if np.isfinite(sd) and sd > 0:
        lo, hi = mu - clip_sigma * sd, mu + clip_sigma * sd
        flux = np.clip(flux, lo, hi)

    x = resampleToFixedLength(time, flux, seq_len=seq_len)

    x = x - np.mean(x)
    std = np.std(x)
    if np.isfinite(std) and std > 0:
        x = x / std

    return x.astype(np.float32)


---

> Create a `torch` dataset class

In [ ]:
class LightCurveDataset(Dataset):
    def __init__(self, df, seq_len=2000):
        self.df = df.reset_index(drop=True)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = preprocessLightCurve(row["time"], row["flux"], seq_len=self.seq_len)

        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)  # (1, T)
        y = torch.tensor(row["label"], dtype=torch.long)
        return x, y


---

> Build a residual TCN for classification

In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)

        self.residual = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.dropout(x)
        return F.relu(x + res)


class TCNClassifier(nn.Module):
    def __init__(self, num_classes, in_ch=1, hidden=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            TCNBlock(in_ch, hidden, dilation=1, dropout=dropout),
            TCNBlock(hidden, hidden * 2, dilation=2, dropout=dropout),
            TCNBlock(hidden * 2, hidden * 4, dilation=4, dropout=dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden * 4, num_classes)

    def forward(self, x):
        # x: (B, 1, T)
        x = self.net(x)
        x = self.pool(x).squeeze(-1)  # (B, C)
        return self.fc(x)


---

> Define the data loaders

In [ ]:
def makeDataLoaders(train_df, val_df, test_df, seq_len=2000, batch_size=32):
    train_ds = LightCurveDataset(train_df, seq_len=seq_len)
    val_ds = LightCurveDataset(val_df, seq_len=seq_len)
    test_ds = LightCurveDataset(test_df, seq_len=seq_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


---

> Define the training loop

In [ ]:
def train(model, train_loader, val_loader, device=device, num_epochs=20, lr=1e-4, class_weights=None):
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    if class_weights is not None:
        class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * y.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)
                loss = criterion(logits, y)

                val_loss += loss.item() * y.size(0)
                pred = logits.argmax(dim=1)
                correct += (pred == y).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)

        print(f"Epoch {epoch+1:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_tcn.pt")


---

> Train-test-validation split

In [ ]:
def splitData(df, test_size=0.2, val_size=0.2, random_state=42):
    train_df, temp_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=df["label"]
    )

    rel_val_size = val_size / (1.0 - test_size)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=rel_val_size,
        random_state=random_state,
        stratify=temp_df["label"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


---

> Model generation and training

In [ ]:
train_df, val_df, test_df = splitData(df, test_size=0.2, val_size=0.2)


In [ ]:
# Define class weights
classes = np.unique(train_df["label"])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)


In [ ]:
# Create data loaders 
seq_len = 2000
train_loader, val_loader, test_loader = makeDataLoaders(
    train_df, val_df, test_df, seq_len=seq_len, batch_size=32
)


In [ ]:
# Define Model
num_classes = len(le.classes_)
model = TCNClassifier(num_classes=num_classes, in_ch=1, hidden=64, dropout=0.1)


In [ ]:
# Train
train(model, train_loader, val_loader, device=device, num_epochs=20, lr=1e-4, class_weights=weights)
